In [ ]:
import pandas as pd

# Load the CSV (make sure it's uploaded to Colab first)
df_merged = pd.read_csv('/content/db_merged.csv')

# Show first few rows
df_merged.head()

,Country,Average Gross,Average Gross (%GDP),publications,citations,Private Investment ($B) (2013-2024),Newly Founded AI Companies (2013-2024),Ai and ML(Popularity),AI_usage_score,AI_usage_percent,...,pct_patents_accepted,accepted_patent_per_capita,Rank,Working_Age_Pop_Using_AI,Median_Broadband _download_speed_(MBPS),Mean_Download_speed_(MBPS),Number_of_SuperComputers,nb_of_computers_in_Top500,Max_performance(TFlops),Computing_power_per_capita
0,US,766897.69300,3.392855,15.386694,27.070086,471,"6,956",18.0,8.295607,48.797686,...,78.978804,0.000873,1,28.3,302.68,161.97,171,172,6475559,0.019429
1,CN,665572.57570,2.402397,21.245047,19.197077,119,"1,605",72.0,11.540426,67.884856,...,45.976365,0.000727,2,16.3,223.47,37.56,40,63,319062,0.000226
2,SG,12073.59353,1.959426,0.844844,1.599431,7,0,35.0,9.783133,57.547838,...,75.680859,0.000350,3,60.9,407.05,134.43,5,0,0,0.000000
3,GB,98225.18148,2.787737,3.885665,7.136482,28,885,16.0,7.704708,45.321813,...,47.159517,0.000103,4,38.9,162.77,110.99,9,14,84814,0.001266
4,FR,79429.81394,2.215881,3.275510,1.973758,11,468,16.0,8.242176,48.483387,...,68.316832,0.000063,5,44.0,346.04,176.97,23,24,298086,0.004400


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

# Load data
df_merged = pd.read_csv('/content/db_merged.csv')

# Target
y = 1 / df_merged['Rank']

# Features
X = df_merged.drop(columns=['Country', 'Rank'])

# Clean column names
X.columns = X.columns.str.replace(' ', '_')
X.columns = X.columns.str.replace('[^A-Za-z0-9_]', '', regex=True)

# Ensure numeric
X = X.apply(pd.to_numeric, errors='coerce')

# Fill missing values
X = X.fillna(X.median())

# Model
model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Leave-One-Out CV
loo = LeaveOneOut()
y_pred = cross_val_predict(model, X, y, cv=loo)

# Evaluation
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
spearman_corr, _ = spearmanr(y, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("Spearman correlation:", spearman_corr)

MAE: 0.05039270041342232
RMSE: 0.11751454210967796
Spearman correlation: 0.8551594746716699


In [ ]:
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print(importances.head(10))

accepted_patent_per_capita               0.310611
publications                             0.153911
Average_Gross                            0.124359
Median_Broadband__download_speed_MBPS    0.074728
total_data_centers_per_capita            0.062208
citations                                0.059451
Private_Investment_B_20132024            0.040499
num_patent_applications                  0.038471
avg_sentiment                            0.030432
Newly_Founded_AI_Companies_20132024      0.019023
dtype: float32


In [ ]:
df_merged[df_merged['Country'] == 'CA']

,Country,Average Gross,Average Gross (%GDP),publications,citations,Private Investment ($B) (2013-2024),Newly Founded AI Companies (2013-2024),Ai and ML(Popularity),AI_usage_score,AI_usage_percent,...,pct_patents_accepted,accepted_patent_per_capita,Rank,Working_Age_Pop_Using_AI,Median_Broadband _download_speed_(MBPS),Mean_Download_speed_(MBPS),Number_of_SuperComputers,nb_of_computers_in_Top500,Max_performance(TFlops),Computing_power_per_capita
7,CA,37188.84818,1.836788,2.027768,4.146488,15,492,22.0,8.171913,48.070075,...,59.466425,0.00032,8,35.0,256.49,152.25,19,0,0,0.0
